In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import seaborn as sns
import pandas as pd
from matplotlib.ticker import PercentFormatter
from matplotlib import pyplot as plt

from analysis.connectors.matomo import MatomoSQLConnector



# popular keywords
matomo = MatomoSQLConnector()
await matomo.connect()


In [ ]:
#interval_start = '2026-01-28 17:00:00'
# interval_stop = '2026-02-20 14:00:00'

#interval_stop = '2026-01-30 00:00:00'


#interval_start = '2026-02-20 17:00:00'
#interval_stop = '2026-12-31 00:00:00'



# Versions

# - 28 janvier / 20 février :
#interval_start = '2026-01-28 17:00:00'
#interval_stop = '2026-02-20 14:00:00'


# - 20 février / 07 avril :
#interval_start = '2026-02-20 17:00:00'
#interval_stop = '2026-04-07 22:00:00'

# - 11 avril / 30 avril :
#interval_start = '2026-04-10 17:00:00'
#interval_stop = '2026-04-29 12:00:00'

# - 5 mai / now
interval_start = '2026-05-05 00:00:00'
interval_stop = '2026-12-31 12:00:00'

In [ ]:
def get_xp(df):
    xp_columns = pd.DataFrame(df['experiments'].apply(eval).tolist())
    return pd.DataFrame({'xp': xp_columns[0].apply(lambda x : x['variation']['name'] if x is not None else None)})

In [ ]:
# ok search event
query =  f"""
        SELECT action_timestamp, idVisit, action_id, action_eventaction, action_eventname, experiments FROM matomo_partitioned
        WHERE action_eventcategory = 'search'
          AND action_timestamp >= '{interval_start}'
          AND action_timestamp < '{interval_stop}'
          ORDER BY action_timestamp asc;
    """
presearch_data = await matomo.run_query(query)
presearch_data_df = pd.DataFrame(presearch_data, columns=['action_timestamp', 'idVisit', 'action_id', 'action_eventaction', 'action_eventname' ,'experiments'])
presearch_event_columns = pd.json_normalize(presearch_data_df['action_eventname'].apply(eval))


presearch_experiments_parsed = get_xp(presearch_data_df)

presearch_data_df = pd.concat([presearch_data_df, presearch_event_columns, presearch_experiments_parsed], axis=1)

#presearch_data_df = pd.concat([presearch_data_df, presearch_event_columns], axis=1)

In [ ]:
# tmp select results events
query =  f"""
        SELECT action_timestamp, idVisit, action_id, action_eventaction, action_eventname, experiments FROM matomo_partitioned
        WHERE action_eventcategory = 'selectResult'
          AND action_timestamp >= '{interval_start}'
          AND action_timestamp < '{interval_stop}'
          ORDER BY action_timestamp asc;
    """
select_res_data = await matomo.run_query(query)
select_df = pd.DataFrame(select_res_data, columns=['action_timestamp', 'idVisit', 'action_id', 'action_eventname', 'experiments', 'truc'])
select_df['action_eventaction'] = 'selectResult'
select_event_columns = pd.json_normalize(select_df['action_eventname'].apply(eval))

select_df = pd.concat([select_df, select_event_columns], axis=1)

In [ ]:
searches = pd.concat([presearch_data_df, select_df]).sort_values(by='action_timestamp', ascending=True)

In [ ]:
presearch_agg = []

count = 0
limit = 500000
#limit = 3
for v, g in searches.groupby(["idVisit"]):
    count = count +1
    if count > limit :
        break
    search = None
    idVisit = v[0]

    for entry in g.to_records(index=False):
        #print(entry)

        entry_type = entry["action_eventaction"]

        if entry_type == "presearch" or  entry_type == "fullsearch":
            query = entry["query"]
            # case new query
            if search == None or query != search['query'] :
                if search != None:
                    presearch_agg.append(search)
                search = {'query':query, 'class': entry['class'], 'idVisit':entry['idVisit'], 'xp':entry['xp']}
        elif entry_type == 'selectPresearchResult':
            if search == None :
                break
            # for now, only keep track of the latest selection
            search['all_results'] = False
            search['selection_algo'] = entry['algo']
            search['selection_url'] = entry['url']
        elif entry_type == 'clickSeeAllResults':
            if search == None :
                break
            search['all_results'] = True
        elif entry_type == 'selectResult':
            if search == None :
                break
            # for now, only keep track of the latest selection
            search['all_results'] = True
            search['selection_algo'] = entry['algo']
            search['selection_url'] = entry['url']

    if search != None:
        presearch_agg.append(search)

presearch_agg_df = pd.DataFrame(presearch_agg)

In [ ]:
def get_source(s):
    if isinstance(s, str):
        if s[0] == '/' :
            return s[1:s[1:].index('/')+1]
        else :
            return 'external'
    else:
        return None

presearch_agg_df['source'] = presearch_agg_df['selection_url'].apply(lambda s : get_source(s)).dropna()

In [ ]:
presearch_agg_df['source'].value_counts().plot.pie(y='classified', figsize=(5, 5),  autopct='%1.1f%%', pctdistance=1.1, labeldistance=1.2)

In [ ]:
import math

def pct(num):
    return round(num*100, 0)

total = presearch_agg_df["class"].dropna().shape[0]

presearch_classes_res = []
for label in ['all'] + presearch_agg_df["class"].dropna().unique().tolist():
    if label == 'all':
        ps_class = presearch_agg_df
    else:
        ps_class = presearch_agg_df[presearch_agg_df["class"] == label]

    count = ps_class.shape[0]

    selections_count = ps_class['selection_algo'].dropna().shape[0]
    count_selections = pct(selections_count / count) if count > 0 else 0

    presearch_algo_count = ps_class[ps_class['selection_algo'] == 'presearch'].shape[0]
    presearch_algo = pct(presearch_algo_count/ selections_count) if selections_count > 0 else 0

    prequa_algo_count = ps_class[ps_class['selection_algo'] == 'pre-qualified'].shape[0]
    prequa_algo = pct(prequa_algo_count/ selections_count) if selections_count > 0 else 0

    text_algo_count = ps_class[ps_class['selection_algo'] == 'fulltext'].shape[0]
    text_algo = pct(text_algo_count/ selections_count) if selections_count > 0 else 0

    #all_results_counts = ps_class[ps_class['all_results'] == True].shape[0]
    #all_results = pct(all_results_counts / count)

    nothing = pct(1 - (selections_count / count)) if count > 0 else 1

    class_share = pct(count/total)

    if (label != 'issue'):
        presearch_classes_res.append({'class': label, 'count_selections': count_selections, 'presearch_algo':presearch_algo, 'prequa_algo':prequa_algo, 'text_algo':text_algo, 'nothing': nothing,  'class_share': class_share, 'count': count}) #, 'ratio': count / queries.shape[0]})

ps_perfs = pd.DataFrame(presearch_classes_res)

In [ ]:

ps_perfs.sort_values(by=['count'], ascending=False, inplace=True)

fig, ax1 = plt.subplots(figsize=(20, 10))

tidy2=ps_perfs[['class_share', 'nothing', 'count_selections', 'presearch_algo', 'prequa_algo', 'text_algo', 'class']].melt(id_vars='class').rename(columns=str.title)
ax = sns.barplot(x='Class', y='Value', hue='Variable', data=tidy2, ax=ax1)
ax.yaxis.set_major_formatter(PercentFormatter())

for cont in ax.containers:
    ax.bar_label(cont)

sns.despine(fig)

In [ ]:
presearch_agg_df[presearch_agg_df['class'] == 'natural']['query'].value_counts()[-10:]

In [ ]:
presearch_agg_df[(presearch_agg_df['class'] == 'unknown') & (presearch_agg_df['selection_algo'].isna())][-300:]['query'].tolist()

In [ ]:
hs = presearch_agg_df[presearch_agg_df['query'] == 'heures supplémentaires']
contribs = hs[hs['selection_algo'] == 'presearch']['selection_url'].apply(lambda x : x.startswith('/contribution/'))
contribs.sum() / contribs.shape[0]

In [ ]:
hs